In [ ]:
from jax import numpy as jnp
from flax import nnx

def softmax_cross_attention(q: jnp.ndarray, k: jnp.ndarray, v: jnp.ndarray, is_causal):
    T_q, T_k = q.shape[-3], k.shape[-3]
    d_head = q.shape[-1]
    
    causal_mask = jnp.where(
        jnp.tri(T_q, T_k) if is_causal else jnp.ones((T_q, T_k)),
        0,
        -jnp.inf
    )

    attn_scores = jnp.einsum(
        "...qhd,...khd->...hqk",
        q,
        k
    ) / d_head ** 0.5 # forgot to scale
    attn_weights = nnx.softmax(attn_scores + causal_mask, axis=-1)

    return jnp.einsum(
        "...hqk,...khd->...qhd", # forgot ...
        attn_weights,
        v
    )


class MultiHeadAttention(nnx.Module):
    def __init__(self, d_model: int, d_head: int, rngs: nnx.Rngs):
        assert d_model % d_head == 0
        n_head = d_model // d_head

        projection = lambda : nnx.Einsum(
            "...e,...ehd->...hd",
            kernel_shape=(d_model, n_head, d_head),
            bias_shape=(n_head, d_head),
            rngs=rngs
        )

        self.W_k = projection()
        self.W_q = projection()
        self.W_v = projection()

        self.W_o = nnx.Einsum(
            "...hd,...hde->...e",
            kernel_shape=(n_head, d_head, d_model),
            bias_shape=(d_model,),
            rngs=rngs
        )

    def __call__(self, X: jnp.ndarray, memory: jnp.ndarray | None = None, is_causal = False):
        if memory is None:
            memory = X

        q = self.W_q(X)
        k = self.W_k(memory)
        v = self.W_v(memory)

        y = softmax_cross_attention(q, k, v, is_causal)

        return self.W_o(y)
